In [ ]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import matplotlib.pyplot as plt

DURATION = 500  # in ms
AMPLITUDE = 0.01
SEED = 100
PATH_ACCURACY = f"/Users/lin/Documents/Bachelor thesis /hallucinations/simulations/results/S_i_S_e/all_{DURATION}_{AMPLITUDE}_{SEED}_accuracy"


In [ ]:
# load data
# accuracies_ctrl = np.load(f"{PATH_ACCURACY}_ctrl.npy")
# accuracies_schz = np.load(f"{PATH_ACCURACY}_schz.npy")
accuracies_ctrl = np.load(f"/Users/lin/Documents/Bachelor thesis /hallucinations/simulations/results/BOLD classification/ampl_0.05/short_sim_schz0_ctrl0_accuracies_ctrl.npy")
accuracies_schz = np.load(f"/Users/lin/Documents/Bachelor thesis /hallucinations/simulations/results/BOLD classification/ampl_0.05/short_sim_schz0_ctrl0_accuracies_schz.npy")

print(f"Control accuracy:       {np.mean(accuracies_ctrl):.7f} ± {np.std(accuracies_ctrl):.3f}")
print(f"Schizophrenia accuracy: {np.mean(accuracies_schz):.7f} ± {np.std(accuracies_schz):.3f}")

# Violin plot comparison
import seaborn as sns
import pandas as pd

# Prepare DataFrame for seaborn
print(len(accuracies_ctrl), len(accuracies_schz))
df = pd.DataFrame({
    'Accuracy':  list(accuracies_ctrl) + list(accuracies_schz),
    'Group': (['Control'] * len(accuracies_ctrl)) +
             (['Schizophrenia'] * len(accuracies_schz))
})
plt.figure(figsize=(8, 6))
sns.violinplot(x='Group', y='Accuracy', data=df, palette='Set2')
plt.title('Classification Accuracy by Group')
plt.ylabel('Accuracy')
plt.grid()
plt.savefig(f"{PATH_ACCURACY}_plot.pdf")
plt.show()

# statistical test
from scipy.stats import ttest_ind
from scipy.stats import ttest_1samp
t_stat_ctrl, p_value_ctrl = ttest_ind(accuracies_ctrl, accuracies_schz)
t_stat_schz, p_value_schz = ttest_1samp(accuracies_schz, 0.5)
t_stat_ctrl, p_value_ctrl = ttest_1samp(accuracies_ctrl, 0.5)
print(f"Control vs Schz: t={t_stat_ctrl:.3f}, p={p_value_ctrl:.3f}")
print(f"Schizophrenia vs Chance: t={t_stat_schz:.3f}, p={p_value_schz:.3f}")


In [ ]:
# load data
# FP_per_subj = np.load(f"{PATH_ACCURACY}_FP.npy", allow_pickle=True).item()
# FN_per_subj = np.load(f"{PATH_ACCURACY}_FN.npy", allow_pickle=True).item()

FP_per_subj = np.load(f"/Users/lin/Documents/Bachelor thesis /hallucinations/simulations/results/BOLD classification/ampl_0.05/short_sim_schz0_ctrl0_FP_per_subj.npy", allow_pickle=True).item()
FN_per_subj = np.load(f"/Users/lin/Documents/Bachelor thesis /hallucinations/simulations/results/BOLD classification/ampl_0.05/short_sim_schz0_ctrl0_FN_per_subj.npy", allow_pickle=True).item()

subjects = FP_per_subj.keys()

# Quantities of false Positives
# accuracies_ctrl = np.array(accuracies_ctrl)
# accuracies_ctrl = accuracies_ctrl.reshape(27,n_splits)
FP_schz = []
FP_ctrl = []
FN_schz = []
FN_ctrl = []

# violin plots of the TP
# 1. average across the splits for each subject
avg_FP_per_subj = {}
for subj in subjects:
    avg_FP_per_subj[subj] = np.mean(FP_per_subj[subj])
    # print(avg_FP_per_subj[subj])

    if 'schz' in subj:
        FP_schz.append(avg_FP_per_subj[subj])
    else:
        FP_ctrl.append(avg_FP_per_subj[subj])

# statistical testing if they are significantly different

from scipy import stats

# Mann-Whitney U test (non-parametric, doesn't assume normality)
stat, p = stats.mannwhitneyu(FP_ctrl, FP_schz, alternative='two-sided')
print(f"Mann-Whitney U: stat={stat:.3f}, p={p:.4f}")

# Also compute effect size (rank-biserial correlation)
n1, n2 = len(FP_ctrl), len(FP_schz)
effect_size = 1 - (2 * stat) / (n1 * n2)
print(f"Effect size (r): {effect_size:.3f}")

# If you want t-test as well for comparison:
stat_t, p_t = stats.ttest_ind(FP_ctrl, FP_schz)
print(f"t-test: stat={stat_t:.3f}, p={p_t:.4f}")

# Visualization using violin plots to compare both groups

import seaborn as sns
import pandas as pd

df_FP = pd.DataFrame({
    'False Positives': np.concatenate([FP_ctrl, FP_schz]),
    'Group': ['Control'] * len(FP_ctrl) + ['Schizophrenia'] * len(FP_schz)
})

plt.figure(figsize=(8, 6))
sns.violinplot(x='Group', y='False Positives', data=df_FP, palette='Set2', )
plt.title('False Positives by Group')
plt.grid()
# plt.savefig(f"{PATH_ACCURACY}_FP_plot.pdf")
plt.show()
